<a href="https://colab.research.google.com/github/syekumar/DSA0302---NATURAL-LANGUAGE-PROCESSING/blob/main/NLP_Coding_Challange.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import sys

def tokenize_text(lines):
    # The regular expression pattern for matching the word tokens
    token_pattern = re.compile(r"[A-Za-z]+(?:['-][A-Za-z]+)*")

    output_lines = []
    for line in lines:
        # Find all non-overlapping matches in the line from left to right
        tokens = token_pattern.findall(line)
        output_lines.append(" ".join(tokens))
    return output_lines

sample_input_lines = [
    "The dog's tail wagged -- twice!",
    "state-of-the-art NLP systems, circa 2026.",
    "12345 ... ???",
    "Don't re-use rock--roll O'Neill's x"
]

print("--- Sample Output ---")
for result in tokenize_text(sample_input_lines):
    print(result)

--- Sample Output ---
The dog's tail wagged twice
state-of-the-art NLP systems circa

Don't re-use rock roll O'Neill's x


In [3]:
def simulate_nfa(input_data):
    lines = [line.strip() for line in input_data.strip().split('\n') if line.strip()]
    if not lines:
        return

    line_idx = 0
    # 1. Number of states n and alphabet size k
    n, k = map(int, lines[line_idx].split())
    line_idx += 1

    # 2. Alphabet symbols
    alphabet = lines[line_idx].split()
    line_idx += 1

    # 3. Start state
    start_state = int(lines[line_idx].strip())
    line_idx += 1

    # 4. Accepting states
    accept_parts = list(map(int, lines[line_idx].split()))
    m = accept_parts[0]
    accepting_states = set(accept_parts[1:])
    line_idx += 1

    # 5. Number of transitions t
    t = int(lines[line_idx].strip())
    line_idx += 1

    # Transitions representation: adj[state][char] = set(next_states)
    transitions = {}
    for _ in range(t):
        parts = lines[line_idx].split()
        p = int(parts[0])
        c = parts[1]
        r = int(parts[2])

        if p not in transitions:
            transitions[p] = {}
        if c not in transitions[p]:
            transitions[p][c] = set()
        transitions[p][c].add(r)
        line_idx += 1

    # 6. Number of query strings Q
    Q = int(lines[line_idx].strip())
    line_idx += 1

    # 7. Process query strings
    for _ in range(Q):
        query = lines[line_idx].strip()
        line_idx += 1

        # '#' denotes the empty string epsilon
        if query == "#":
            query = ""

        # Set of active states initially
        current_states = {start_state}

        for char in query:
            next_states = set()
            for state in current_states:
                if state in transitions and char in transitions[state]:
                    next_states.update(transitions[state][char])
            current_states = next_states
            if not current_states:
                break

        if current_states.intersection(accepting_states):
            print("ACCEPT")
        else:
            print("REJECT")

# Test with the sample input provided
sample_input = """
5 3
b a !
0
1 4
5
0 b 1
1 a 2
2 a 3
3 a 3
3 ! 4
5
baa!
baaa!
ba!
b
baa
"""

simulate_nfa(sample_input)

ACCEPT
ACCEPT
REJECT
REJECT
REJECT


In [5]:
import sys

def is_vowel(char):
    return char in "aeiou"

def count_vowel_groups(stem):
    groups = 0
    in_vowel = False
    for char in stem:
        if is_vowel(char):
            if not in_vowel:
                groups += 1
                in_vowel = True
        else:
            in_vowel = False
    return groups

def is_doublable(stem):
    if len(stem) < 3:
        return False
    c1, v1, c2 = stem[-3], stem[-2], stem[-1]
    if is_vowel(c1) or not is_vowel(v1) or is_vowel(c2):
        return False
    if c2 in "wxy":
        return False
    if count_vowel_groups(stem) != 1:
        return False
    return True

def apply_inflection(stem, feature):
    if feature in ("N+PL", "V+3SG"):
        if stem.endswith(("s", "x", "z", "ch", "sh")):
            return stem + "es"
        elif (
            len(stem) >= 2
            and not is_vowel(stem[-1])
            and stem[-1] == "o"
            and not is_vowel(stem[-2])
        ):
            return stem + "es"
        elif len(stem) >= 2 and stem[-1] == "y" and not is_vowel(stem[-2]):
            return stem[:-1] + "ies"
        else:
            return stem + "s"
    elif feature == "V+PROG":
        if len(stem) >= 3 and stem[-1] == "e" and not is_vowel(stem[-2]):
            return stem[:-1] + "ing"
        elif is_doublable(stem):
            return stem + stem[-1] + "ing"
        else:
            return stem + "ing"
    elif feature in ("V+PAST", "ADJ+COMP", "ADJ+SUP"):
        suffix_map = {
            "V+PAST": ("d", "ed"),
            "ADJ+COMP": ("r", "er"),
            "ADJ+SUP": ("st", "est"),
        }
        sfx_short, sfx_long = suffix_map[feature]
        if stem.endswith("e"):
            return stem + sfx_short
        elif len(stem) >= 2 and stem[-1] == "y" and not is_vowel(stem[-2]):
            return stem[:-1] + "i" + sfx_long
        elif is_doublable(stem):
            return stem + stem[-1] + sfx_long
        else:
            return stem + sfx_long
    return stem

def process_morphology(input_lines):
    idx = 0
    L = int(input_lines[idx].strip())
    idx += 1
    lexicon = {}
    for _ in range(L):
        stem, feature, form = input_lines[idx].split()
        lexicon[(stem, feature)] = form
        idx += 1
    Q = int(input_lines[idx].strip())
    idx += 1
    output_results = []
    for _ in range(Q):
        stem, feature = input_lines[idx].split()
        idx += 1
        if (stem, feature) in lexicon:
            output_results.append(lexicon[(stem, feature)])
        else:
            output_results.append(apply_inflection(stem, feature))
    return output_results

def solve_problem_3():
    input_data = sys.stdin.read().splitlines()
    if not input_data:
        return
    for result in process_morphology(input_data):
        print(result)

if __name__ == "__main__":
    sample_input_lines = [
        "3",
        "goose N+PL geese",
        "go V+PAST went",
        "good ADJ+COMP better",
        "8",
        "cat N+PL",
        "church N+PL",
        "baby N+PL",
        "stop V+PROG",
        "make V+PROG",
        "carry V+PAST",
        "goose N+PL",
        "go V+PAST",
    ]
    print("--- Sample Output ---")
    for result in process_morphology(sample_input_lines):
        print(result)

--- Sample Output ---
cats
churches
babies
stopping
making
carried
geese
went


In [6]:
def parse_morphology(input_data):
    lines = [line.strip() for line in input_data.strip().split('\n') if line.strip()]
    if not lines:
        return

    idx = 0
    S = int(lines[idx].strip())
    idx += 1

    lexicon = []
    for _ in range(S):
        stem, pos = lines[idx].split()
        lexicon.append((stem, pos))
        idx += 1

    Q = int(lines[idx].strip())
    idx += 1

    # Mapping of POS to their allowed grammatical features
    pos_features = {
        'N': ['PL'],
        'V': ['3SG', 'PROG', 'PAST'],
        'ADJ': ['COMP', 'SUP']
    }

    # Helper functions from Problem 3
    def is_vowel(char):
        return char in "aeiou"

    def count_vowel_groups(stem_str):
        groups = 0
        in_vowel = False
        for char in stem_str:
            if is_vowel(char):
                if not in_vowel:
                    groups += 1
                    in_vowel = True
            else:
                in_vowel = False
        return groups

    def is_doublable(stem_str):
        if len(stem_str) < 3:
            return False
        c1, v1, c2 = stem_str[-3], stem_str[-2], stem_str[-1]
        if is_vowel(c1) or not is_vowel(v1) or is_vowel(c2):
            return False
        if c2 in "wxy":
            return False
        if count_vowel_groups(stem_str) != 1:
            return False
        return True

    def generate_form(stem_str, pos, feature):
        # N+PL / V+3SG
        if (pos == 'N' and feature == 'PL') or (pos == 'V' and feature == '3SG'):
            if stem_str.endswith(("s", "x", "z", "ch", "sh")):
                return stem_str + "es"
            elif (
                len(stem_str) >= 2
                and not is_vowel(stem_str[-1])
                and stem_str[-1] == "o"
                and not is_vowel(stem_str[-2])
            ):
                return stem_str + "es"
            elif len(stem_str) >= 2 and stem_str[-1] == "y" and not is_vowel(stem_str[-2]):
                return stem_str[:-1] + "ies"
            else:
                return stem_str + "s"

        # V+PROG
        elif pos == 'V' and feature == 'PROG':
            if len(stem_str) >= 3 and stem_str[-1] == "e" and not is_vowel(stem_str[-2]):
                return stem_str[:-1] + "ing"
            elif is_doublable(stem_str):
                return stem_str + stem_str[-1] + "ing"
            else:
                return stem_str + "ing"

        # V+PAST, ADJ+COMP, ADJ+SUP
        elif (pos == 'V' and feature == 'PAST') or (pos == 'ADJ' and feature in ('COMP', 'SUP')):
            suffix_map = {
                "PAST": ("d", "ed"),
                "COMP": ("r", "er"),
                "SUP": ("st", "est"),
            }
            sfx_short, sfx_long = suffix_map[feature]
            if stem_str.endswith("e"):
                return stem_str + sfx_short
            elif len(stem_str) >= 2 and stem_str[-1] == "y" and not is_vowel(stem_str[-2]):
                return stem_str[:-1] + "i" + sfx_long
            elif is_doublable(stem_str):
                return stem_str + stem_str[-1] + sfx_long
            else:
                return stem_str + sfx_long
        return ""

    # Process each surface form query
    for _ in range(Q):
        surface_form = lines[idx].strip()
        idx += 1

        analyses = []

        # Try matching each lexicon entry
        for stem, pos in lexicon:
            # Check 1: Is it a bare stem?
            if surface_form == stem:
                analyses.append(f"{stem}+{pos}")

            # Check 2: Try inflecting this stem with each permitted feature of its POS
            features = pos_features.get(pos, [])
            for feature in features:
                if generate_form(stem, pos, feature) == surface_form:
                    analyses.append(f"{stem}+{pos}+{feature}")

        if analyses:
            # Sort lexicographically and print
            analyses.sort()
            print(", ".join(analyses))
        else:
            print("UNKNOWN")

# Sample run with given sample input
sample_input_p4 = """
5
cat N
watch N
watch V
study V
big ADJ
8
cats
watches
watching
studied
bigger
cat
biggest
elephants
"""

print("--- Sample Output ---")
parse_morphology(sample_input_p4)

--- Sample Output ---
cat+N+PL
watch+N+PL, watch+V+3SG
watch+V+PROG
study+V+PAST
big+ADJ+COMP
cat+N
big+ADJ+SUP
UNKNOWN


In [8]:
import sys

def porter_is_consonant(word, i):
    char = word[i]
    if char in "aeiou":
        return False
    if char == "y":
        if i == 0:
            return True
        else:
            return not porter_is_consonant(word, i - 1)
    return True

def get_porter_measure(stem):
    n = len(stem)
    i = 0
    while i < n and porter_is_consonant(stem, i):
        i += 1
    m = 0
    while True:
        v_start = i
        while i < n and not porter_is_consonant(stem, i):
            i += 1
        if i == v_start:
            break
        c_start = i
        while i < n and porter_is_consonant(stem, i):
            i += 1
        if i == c_start:
            break
        m += 1
    return m

def porter_has_vowel(stem):
    for i in range(len(stem)):
        if not porter_is_consonant(stem, i):
            return True
    return False

def porter_ends_double_consonant(stem):
    if (
        len(stem) >= 2
        and stem[-1] == stem[-2]
        and porter_is_consonant(stem, len(stem) - 1)
    ):
        return True
    return False

def porter_ends_cvc(stem):
    if len(stem) >= 3:
        c1 = porter_is_consonant(stem, len(stem) - 3)
        v = not porter_is_consonant(stem, len(stem) - 2)
        c2 = porter_is_consonant(stem, len(stem) - 1)
        if c1 and v and c2 and stem[-1] not in "wxy":
            return True
    return False

def porter_step_1(word):
    if word.endswith("sses"):
        word = word[:-2]
    elif word.endswith("ies"):
        word = word[:-2]
    elif word.endswith("ss"):
        pass
    elif word.endswith("s"):
        word = word[:-1]

    ed_ing_removed = False
    if word.endswith("eed"):
        stem = word[:-3]
        if get_porter_measure(stem) > 0:
            word = stem + "ee"
    elif word.endswith("ed"):
        stem = word[:-2]
        if porter_has_vowel(stem):
            word = stem
            ed_ing_removed = True
    elif word.endswith("ing"):
        stem = word[:-3]
        if porter_has_vowel(stem):
            word = stem
            ed_ing_removed = True

    if ed_ing_removed:
        if word.endswith("at") or word.endswith("bl") or word.endswith("iz"):
            word = word + "e"
        elif porter_ends_double_consonant(word) and word[-1] not in "lsz":
            word = word[:-1]
        elif get_porter_measure(word) == 1 and porter_ends_cvc(word):
            word = word + "e"

    if word.endswith("y"):
        stem = word[:-1]
        if porter_has_vowel(stem):
            word = stem + "i"
    return word

def process_porter(input_lines):
    Q = int(input_lines[0].strip())
    output_results = []
    for i in range(1, Q + 1):
        if i < len(input_lines):
            word = input_lines[i].strip()
            output_results.append(porter_step_1(word))
    return output_results

def solve_problem_5():
    input_data = sys.stdin.read().splitlines()
    if not input_data:
        return
    for result in process_porter(input_data):
        print(result)

if __name__ == "__main__":
    sample_input_lines = [
        "8",
        "caresses",
        "ponies",
        "cats",
        "feed",
        "agreed",
        "motoring",
        "hopping",
        "happy",
    ]
    print("--- Sample Output ---")
    for result in process_porter(sample_input_lines):
        print(result)

--- Sample Output ---
caress
poni
cat
feed
agree
motor
hop
happi
